# Train "hey Clem" — Clem's custom wake word

Trains an openWakeWord model that fires on **"hey Clem"**, **"okay Clem"**, or
**"hello Clem"**, to replace the placeholder `hey_jarvis` standby wake word.

Built 2026-08-01; "hello clem" added after phoneme measurement — 8 phones,
parity with the known-good "hey jarvis". The training
config is embedded below — no files to upload.

**Before running:**
1. `Runtime → Change runtime type → T4 GPU` (free tier is fine)
2. `Runtime → Run all`
3. Keep this tab open. Total time ≈ 2 hours on a T4 (roughly 30 min of data
   downloads, then clip generation and training).

**Output:** your browser downloads `hey_clem_outputs.zip` at the end. It holds
`hey_clem.onnx` (**the one we deploy** — the robot cannot run .tflite), the
training config, and held-out synthetic test clips for the deployment gate.

This notebook only *trains* the model. Deployment is gated on measuring the
false-accept rate against real room audio with `wakeword/check_model.py` —
do not skip that step.


In [ ]:
# Cell 1 — Environment report. FAIL FAST if there's no GPU.
import sys, shutil, subprocess
print("Python:", sys.version)
if shutil.which("nvidia-smi"):
    print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"],
                         capture_output=True, text=True).stdout)
    GPU = True
else:
    GPU = False
    print("*" * 70)
    print("NO GPU DETECTED. Sample generation measured at ~4 clips/s on CPU —")
    print("that is ~5 HOURS for the 75k clips this config asks for, vs ~15 min")
    print("on a T4. Stop now: Runtime -> Change runtime type -> T4 GPU -> Run all.")
    print("*" * 70)
assert GPU, "Switch to a GPU runtime before continuing (Runtime -> Change runtime type)."


In [ ]:
# Cell 2 — Tooling (runbook step 1).
#
# NOTE the fork: openwakeword.train does `from generate_samples import
# generate_samples`, and only dscripka/piper-sample-generator has that module
# at the top level (rhasspy's upstream restructured it away). The voice model
# checkpoint still comes from rhasspy's release, as in the runbook.
!git clone https://github.com/dscripka/piper-sample-generator
# The fork's generate_samples defaults to models/en-us-libritts-high.pt and
# REQUIRES the matching .pt.json the repo ships — so it gets the v1.0.0 HIGH
# checkpoint its own README documents (train.py 0.6.0 passes no model arg;
# the v2.0.0 medium_r model from the earlier runbook draft has no json here
# and would FileNotFoundError).
!wget -q -O piper-sample-generator/models/en-us-libritts-high.pt https://github.com/rhasspy/piper-sample-generator/releases/download/v1.0.0/en-us-libritts-high.pt

# The fork phonemizes via espeak_phonemizer (ctypes onto the system
# espeak-ng library), NOT piper-phonemize — both halves installed here.
!apt-get -qq -y install espeak-ng > /dev/null
!pip install -q espeak-phonemizer

# openwakeword MUST be --no-deps on py3.12 Colab: v0.6.0 hard-requires
# tflite-runtime on Linux, which has no cp312 build — a plain install makes
# pip silently backtrack to a fossil openwakeword without download_models
# (this killed run #2). Same install pattern as the robot's apps_venv.
# Its real runtime deps are installed explicitly on the next line.
!pip install -q --no-deps openwakeword
# GPU onnxruntime: feature computation runs through openwakeword's ONNX
# models and takes hours on Colab's CPU vs minutes on the T4. onnxscript:
# required by modern torch's ONNX exporter (verified locally).
!pip uninstall -q -y onnxruntime
# ==1.22.*: the CUDA-12 build line. Latest ort-gpu targets CUDA 13, which
# Colab (CUDA 12.8) rejects at session creation -> silent CPU fallback and
# an augment phase measured at ~2 h instead of minutes (run #7).
!pip install -q "onnxruntime-gpu==1.22.*" onnxscript scikit-learn requests scipy tqdm soundfile
!pip install -q webrtcvad || pip install -q webrtcvad-wheels
# Python 3.12 Colab: upstream piper-phonemize ships no cp312 wheel (this
# killed the first run). piper-phonemize-fix 1.2.2 is a verified drop-in
# rebuild — same `piper_phonemize` import, espeak-ng data bundled.
!pip install -q piper-phonemize-fix
!pip install -q mutagen==1.47.0 torchinfo==1.8.0 torchmetrics==1.2.0 speechbrain==0.5.14 audiomentations==0.33.0 torch-audiomentations==0.11.0 acoustics==0.2.6 pronouncing==0.2.0 deep-phonemizer==0.0.19
# The official notebook pins datasets==2.14.6, which also breaks on py3.12
# (it needs pyarrow<14, and pyarrow<14 has no cp312 wheel). 2.21.0 is the
# last mature 2.x: clean with modern pyarrow, still runs fma's loader script.
!pip install -q "datasets==2.21.0"
# Deliberately NOT installed: tensorflow-cpu==2.8.1 / tensorflow_probability /
# onnx_tf from the official notebook. They exist only for .tflite export, which
# we don't use (the robot has no tflite-runtime for Python 3.12) — and those
# pins no longer install on current Colab Python anyway.

# openWakeWord base models (melspectrogram + embedding), per the runbook.
from importlib.metadata import version
import openwakeword.utils as oww_utils
print("openwakeword", version("openwakeword"))
assert hasattr(oww_utils, "download_models"), (
    "An old openwakeword is loaded in this runtime. Use a FRESH runtime: "
    "Runtime -> Disconnect and delete runtime, then Runtime -> Run all.")
oww_utils.download_models()

# Does CUDA actually engage? (Provider being LISTED is not enough — run #7
# listed it and still fell back to CPU.) ort dlopens cuDNN/cuBLAS at session
# creation; torch's pip deps ship those libs, so aim LD_LIBRARY_PATH at them.
import glob as _glob
_nvlibs = ":".join(sorted(_glob.glob("/usr/local/lib/python3.12/dist-packages/nvidia/*/lib")))
import os as _os
_os.environ["NV_LIBS"] = _nvlibs
!export LD_LIBRARY_PATH=$NV_LIBS:$LD_LIBRARY_PATH && python -c "import onnxruntime as o, openwakeword, os; m = os.path.join(os.path.dirname(openwakeword.__file__), 'resources', 'models', 'melspectrogram.onnx'); s = o.InferenceSession(m, providers=['CUDAExecutionProvider', 'CPUExecutionProvider']); print('ORT session providers:', s.get_providers())"
# 'CUDAExecutionProvider' first in that list = GPU features (fast path).
# CPU-only = it still works, just slower; don't stop the run for it.

# Compat shim for the trainer (both faults reproduced and fixed locally,
# 2026-08-01): speechbrain 0.5.14 calls torchaudio.set_audio_backend
# (removed in torchaudio 2.2+), and acoustics 0.2.6 imports
# scipy.special.sph_harm (removed in modern scipy). openwakeword only uses
# acoustics for noise generation, so an import-satisfying alias is safe.
from pathlib import Path
Path("oww_train_shim.py").write_text('''# Compat shim for running openwakeword.train on a modern (2026) stack.
# Each patch below restores behavior an upstream library still expects but
# its dependency removed; all were reproduced and verified locally.
#
# 1. speechbrain 0.5.14 calls torchaudio.set_audio_backend (removed in
#    torchaudio 2.2+; backend dispatch is automatic now) -> no-op.
# 2. acoustics 0.2.6 imports scipy.special.sph_harm (removed in modern
#    scipy; renamed sph_harm_y with swapped args) -> alias. openwakeword
#    uses acoustics only for noise generation and never calls the
#    directivity code, so the alias is import-satisfying, not load-bearing.
# 3. torch >=2.6 defaults torch.load(weights_only=True); the Piper voice
#    checkpoint is a full pickled model from rhasspy's official release
#    (trusted source) -> default weights_only=False, the pre-2.6 behavior.
import torchaudio

if not hasattr(torchaudio, "set_audio_backend"):
    torchaudio.set_audio_backend = lambda *a, **k: None
if not hasattr(torchaudio, "get_audio_backend"):
    torchaudio.get_audio_backend = lambda: "soundfile"

import scipy.special

if not hasattr(scipy.special, "sph_harm"):
    def _sph_harm(m, n, theta, phi, *args, **kwargs):
        return scipy.special.sph_harm_y(n, m, phi, theta, *args, **kwargs)
    scipy.special.sph_harm = _sph_harm

import torch

_orig_torch_load = torch.load

def _torch_load(*args, **kwargs):
    kwargs.setdefault("weights_only", False)
    return _orig_torch_load(*args, **kwargs)

torch.load = _torch_load

# torchaudio 2.11 removed the legacy torchaudio.info; torch_audiomentations
# and openwakeword.data still call it. Minimal old-API surface via soundfile.
if not hasattr(torchaudio, "info"):
    import soundfile as _sf

    class _AudioMetaData:
        def __init__(self, i):
            self.sample_rate = int(i.samplerate)
            self.num_frames = int(i.frames)
            self.num_channels = int(i.channels)
            self.bits_per_sample = 16
            self.encoding = "PCM_S"

    torchaudio.info = lambda p, *a, **k: _AudioMetaData(_sf.info(str(p)))

# torchaudio 2.11's load() delegates to the new torchcodec package and
# raises if it's absent. All audio in this pipeline is plain wav, so fall
# back to a soundfile-backed load with the legacy return convention
# (float32 tensor, channels-first) only when torchcodec is missing.
try:
    import torchcodec  # noqa: F401
except Exception:
    import soundfile as _sf2
    import torch as _torch

    def _sf_load(path, frame_offset=0, num_frames=-1, normalize=True,
                 channels_first=True, **_kwargs):
        data, sr = _sf2.read(str(path), dtype="float32", always_2d=True,
                             start=int(frame_offset),
                             frames=int(num_frames) if num_frames and int(num_frames) > 0 else -1)
        tensor = _torch.from_numpy(data.T if channels_first else data)
        return tensor, sr

    torchaudio.load = _sf_load

import runpy
runpy.run_module("openwakeword.train", run_name="__main__")
''')
!python oww_train_shim.py --help > /dev/null && echo "train.py imports OK (shim active)"

# Preflight (~30 s, throwaway subprocess so the kernel keeps its RAM):
# exercises the exact pieces the long cells need — espeak phonemization
# and the voice-checkpoint unpickle — so failures land HERE, not 30
# minutes into a training cell.
Path("preflight.py").write_text('''import sys
sys.path.insert(0, "piper-sample-generator")
import torch
_o = torch.load
torch.load = lambda *a, **k: _o(*a, **{**k, "weights_only": False})
from espeak_phonemizer import Phonemizer
ipa = Phonemizer("en-us").phonemize("hello clem")
assert "kl" in ipa, f"unexpected phonemes: {ipa}"
print("espeak OK:", ipa)
torch.load("piper-sample-generator/models/en-us-libritts-high.pt", map_location="cpu")
print("PREFLIGHT OK: phonemizer + voice checkpoint load")
''')
!python preflight.py
print("Tooling ready.")


In [ ]:
# Cell 3 — Negative/background data from HuggingFace + canonical sources.
#
# v4 rework, verified against the live repos on 2026-08-01:
#  * AudioSet's tar files are GONE — the repo converted to parquet shards.
#    We stream config "balanced" / split "train" through the same datasets
#    code path the MIT RIRs cell already uses (proven in run #3).
#  * FMA's loader script cannot stream under modern datasets/fsspec, so the
#    music comes straight from FMA's canonical source zip (os.unil.cloud
#    .switch.ch, 7.15 GiB — the exact URL the loader itself uses), with
#    Colab's ffmpeg converting a ~3 h slice.
# Every guard below is CONTENT-based (file counts / exact byte sizes), so
# re-running this cell repairs a half-filled directory instead of skipping it.
import os, subprocess
import numpy as np
import scipy.io.wavfile
import datasets
from pathlib import Path
from tqdm import tqdm


def wav_count(d):
    return len(list(Path(d).glob("*.wav"))) if os.path.isdir(d) else 0


# --- MIT room impulse responses -> ./mit_rirs (270 clips)
output_dir = "./mit_rirs"
if wav_count(output_dir) < 250:
    os.makedirs(output_dir, exist_ok=True)
    rir_dataset = datasets.load_dataset(
        "davidscripka/MIT_environmental_impulse_responses", split="train", streaming=True)
    for row in tqdm(rir_dataset, desc="MIT RIRs"):
        name = row["audio"]["path"].split("/")[-1]
        scipy.io.wavfile.write(os.path.join(output_dir, name), 16000,
                               (row["audio"]["array"] * 32767).astype(np.int16))

# --- AudioSet ambient clips -> ./audioset_16k (2000 x 10 s ~= 5.5 h)
output_dir = "./audioset_16k"
N_AUDIOSET = 2000
if wav_count(output_dir) < N_AUDIOSET:
    os.makedirs(output_dir, exist_ok=True)
    aset = datasets.load_dataset("agkphysics/AudioSet", "balanced",
                                 split="train", streaming=True)
    aset = aset.cast_column("audio", datasets.Audio(sampling_rate=16000))
    done = 0
    for row in tqdm(aset, total=N_AUDIOSET, desc="AudioSet -> 16k"):
        stem = str(row.get("video_id") or f"audioset_{done:05d}")
        scipy.io.wavfile.write(os.path.join(output_dir, stem + ".wav"), 16000,
                               (row["audio"]["array"] * 32767).astype(np.int16))
        done += 1
        if done >= N_AUDIOSET:
            break

# --- FMA music -> ./fma (360 x 30 s ~= 3 h) from the canonical source zip
output_dir = "./fma"
N_FMA = 360
if wav_count(output_dir) < 300:
    os.makedirs(output_dir, exist_ok=True)
    if not (os.path.exists("fma_small.zip")
            and os.path.getsize("fma_small.zip") == 7679594875):
        !wget -c -O fma_small.zip https://os.unil.cloud.switch.ch/fma/fma_small.zip
    # first 8 track-id folders ~= 400 mp3s; plenty for the 3 h slice
    !unzip -q -o fma_small.zip "fma_small/00[0-7]/*" -d fma_zip
    mp3s = sorted(Path("fma_zip").glob("**/*.mp3"))[:N_FMA]
    for mp3 in tqdm(mp3s, desc="FMA -> 16k"):
        out = os.path.join(output_dir, mp3.stem + ".wav")
        subprocess.run(["ffmpeg", "-hide_banner", "-loglevel", "error", "-y",
                        "-i", str(mp3), "-ac", "1", "-ar", "16000", out], check=False)

# --- Precomputed openWakeWord features (sizes verified via HTTP HEAD;
#     wget -c resumes a partial download instead of starting over)
FEATURES = {
    "openwakeword_features_ACAV100M_2000_hrs_16bit.npy": 17280000128,
    "validation_set_features.npy": 184836608,
}
for fname, fsize in FEATURES.items():
    if not (os.path.exists(fname) and os.path.getsize(fname) == fsize):
        url = ("https://huggingface.co/datasets/davidscripka/openwakeword_features"
               f"/resolve/main/{fname}")
        !wget -c {url}
        assert os.path.getsize(fname) == fsize, f"{fname}: incomplete download, run this cell again"

print("Data ready:",
      wav_count("mit_rirs"), "RIRs |",
      wav_count("audioset_16k"), "AudioSet clips |",
      wav_count("fma"), "FMA clips |",
      "features OK")


In [ ]:
%%writefile hey_clem.yaml
# openWakeWord training config for Clem's own wake word.
#
# Run with:  python -m openwakeword.train --training_config hey_clem.yaml \
#                --generate_clips --augment_clips --train_model
# On a modern stack (torchaudio >=2.2 / scipy >=1.17, e.g. 2026 Colab) run it
# through wakeword's oww_train_shim.py instead — hey_clem_colab.ipynb writes
# it — which restores two removed APIs that speechbrain 0.5.14 and acoustics
# 0.2.6 still import. See README.md for the full runbook and expected data.

model_name: "friday"

# Three phrases, one binary model — it activates on any of them. All were
# checked with phrase_check.py / espeak-ng: "clem" renders as klˈɛm, so the
# TTS training data says the name correctly and no creative spelling is needed.
#
# "hello clem" (həlˈoʊ klˈɛm, added 2026-08-01) is the
# robustness anchor: 8 phones, the same count as the known-good "hey jarvis"
# reference. "hey clem" stays because it is what one naturally blurts at the
# robot; "okay clem" measured closest to the reference in clip length.
#
# "clem" on its own is deliberately absent. Measured median TTS length is
# 0.46 s against 0.79 s for "hey jarvis"; a wake word that short carries too
# little acoustic information to separate from ordinary speech, and the price
# is paid in false accepts, which for Clem means opening the cloud voice
# connection unbidden.
target_phrase:
  - "friday"

# Near-misses to train against, on top of the phoneme-overlap negatives
# openWakeWord generates automatically. Each differs from klˈɛm by about one
# feature, which is exactly where a short wake word gets into trouble.
custom_negative_phrases:
  - "fried egg"
  - "fried"
  - "fry day"
  - "fly day"
  - "free day"
  - "Friday night"
  - "Friday morning"
  - "Friday afternoon"
  - "Friday evening"
  - "Friday today"
  - "Friday tomorrow"
  - "Friday's"
  - "Fridays"
  - "right away"
  - "by Friday"
  - "next Friday"
  - "last Friday"

# Above the 20,000 the upstream example calls a minimum, because a short wake
# phrase needs the extra coverage. n_samples is the TOTAL across all target
# phrases (train.py passes the whole phrase list to one generate_samples call),
# so 75,000 keeps the original 25,000-per-phrase density now that there are
# three phrases. On a T4 this is roughly 15 minutes of generation; on 4 CPU
# cores it is closer to 5 hours (measured: ~4 clips/s).
n_samples: 75000
n_samples_val: 7500

tts_batch_size: 50
augmentation_batch_size: 16

# Each generated clip is augmented twice — cheaper robustness than generating
# twice as many clips, since the augmentation itself is random.
augmentation_rounds: 2

# dscripka's fork, NOT rhasspy/piper-sample-generator: train.py does
# `from generate_samples import generate_samples`, and only the fork has that
# module at the top level.
piper_sample_generator_path: "./piper-sample-generator"

output_dir: "./hey_clem_model"

rir_paths:
  - "./mit_rirs"

background_paths:
  - "./audioset_16k"
  - "./fma"

background_paths_duplication_rate:
  - 1
  - 1

false_positive_validation_data_path: "./validation_set_features.npy"

feature_data_files:
  "ACAV100M_sample": "./openwakeword_features_ACAV100M_2000_hrs_16bit.npy"

batch_n_per_class:
  "ACAV100M_sample": 1024
  "adversarial_negative": 50
  "positive": 50

model_type: "dnn"
layer_size: 32

steps: 50000

max_negative_weight: 1500

# Stricter than the upstream default of 0.2/hr is not necessary, but do not
# loosen it: Clem's wake word runs all day in a room with a television.
target_false_positives_per_hour: 0.2


In [ ]:
# Cell 5 — Generate 75k synthetic clips of "hey clem" / "okay clem" / "hello clem"
# (~15 min on T4).
!export LD_LIBRARY_PATH=$NV_LIBS:$LD_LIBRARY_PATH && python oww_train_shim.py --training_config hey_clem.yaml --generate_clips


In [ ]:
# Cell 6 — Augment the clips (reverb from the RIRs, noise from AudioSet/FMA; 2 rounds).
# A crashed augment leaves a PARTIAL features file, and the step's own guard
# is existence-based — it would silently skip and feed garbage to training.
# If the four feature files aren't all present, clear the stragglers.
import glob, os
feats = glob.glob("hey_clem_model/hey_clem/*_features_*.npy")
if 0 < len(feats) < 4:
    for f in feats:
        os.remove(f)
    print("cleared partial features from a previous attempt:", feats)
!export LD_LIBRARY_PATH=$NV_LIBS:$LD_LIBRARY_PATH && python oww_train_shim.py --training_config hey_clem.yaml --augment_clips


In [ ]:
# Cell 7 — Train (~30-40 min on T4).
#
# EXPECTED at the very end: "ModuleNotFoundError: No module named 'onnx_tf'".
# The 0.6.0 wheel attempts tflite conversion unconditionally AFTER writing
# hey_clem.onnx (verified locally); we don't install its ancient toolchain
# because the robot runs ONNX. The error is harmless — run the next cell.
!export LD_LIBRARY_PATH=$NV_LIBS:$LD_LIBRARY_PATH && python oww_train_shim.py --training_config hey_clem.yaml --train_model


In [ ]:
# Cell 8 — Verify + package. Downloads hey_clem_outputs.zip.
import hashlib, os, random, shutil
from pathlib import Path

onnx_candidates = sorted(Path(".").glob("**/hey_clem*.onnx"))
assert onnx_candidates, (
    "No hey_clem .onnx found. Scroll up to the training cell — if training "
    "itself failed, the error is there. Contents of ./hey_clem_model: "
    + str(sorted(str(p) for p in Path("hey_clem_model").glob("**/*"))[:50]))
onnx_path = onnx_candidates[0]
sha = hashlib.sha256(onnx_path.read_bytes()).hexdigest()
print(f"Model: {onnx_path}  ({onnx_path.stat().st_size/1024:.0f} KiB)")
print(f"SHA256: {sha}")

stage = Path("hey_clem_outputs")
if stage.exists():
    shutil.rmtree(stage)
stage.mkdir()
shutil.copy(onnx_path, stage / "hey_clem.onnx")
# Modern torch.onnx externalizes weights into a .data sidecar — without it
# the .onnx is just a graph skeleton (this bit run #8's zip).
sidecar = Path(str(onnx_path) + ".data")
if sidecar.exists():
    shutil.copy(sidecar, stage / "hey_clem.onnx.data")
    print(f"weight sidecar included: {sidecar} ({sidecar.stat().st_size/1024:.0f} KiB)")
shutil.copy("hey_clem.yaml", stage / "hey_clem.yaml")
(stage / "SHA256.txt").write_text(f"{sha}  hey_clem.onnx\n")
for tfl in sorted(Path(".").glob("**/hey_clem*.tflite"))[:1]:
    shutil.copy(tfl, stage / tfl.name)  # not used on the robot; kept for the record

# Held-out synthetic positives for check_model.py --positives (best effort:
# a random sample of the *test* split clips, which the training loop never saw).
heldout = stage / "heldout_synthetic_positives"
test_clips = []
# positive_test ONLY — "*test*" also matches negative_test (adversarial
# near-misses), which salted run #8's "positives" and read as 0%% recall.
for pattern in ("hey_clem_model/**/positive_test/*.wav",
                "hey_clem_model/**/positive_test/**/*.wav"):
    test_clips.extend(Path(".").glob(pattern))
test_clips = sorted(set(test_clips))
if test_clips:
    heldout.mkdir()
    for clip in random.Random(0).sample(test_clips, min(150, len(test_clips))):
        shutil.copy(clip, heldout / clip.name)
    print(f"Held-out synthetic positives: {min(150, len(test_clips))} clips")
else:
    print("No test-split clips found to hold out — fine; real recordings of "
          "you saying the phrase serve as the gate's positives.")

shutil.make_archive("hey_clem_outputs", "zip", ".", "hey_clem_outputs")
print("\nDONE. Downloading hey_clem_outputs.zip ...")
print("Deploy the .onnx ONLY (the robot has no tflite runtime).")
print("Next: get this zip to Claude on the Mac for the check_model.py gate.")
from google.colab import files
files.download("hey_clem_outputs.zip")
